In [2]:
import os
import sys 
print("Active Environment Path:", sys.executable)
from pathlib import Path
import json
import subprocess
import importlib
import tempfile

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, PrecisionRecallDisplay
import shapely

import pystac_client
import planetary_computer
import odc.stac
import rasterio.features

# local
os.chdir("/home/azureuser/EBRD_Tillage/AK_Experiments/AK_pipeline")
sys.path.append(os.getcwd())
from Generic.addSentinelIndices import calculateSentinelIndex, SENTINEL_INDEX_DEPENDENCIES
from utils.classify_tillage import classify_tillage
from ml.nominal_classification import RFC 

Active Environment Path: /anaconda/envs/geospatial-base/bin/python


In [3]:
# ==============================================================================
# 1. HELPER FUNCTIONS FOR IN-MEMORY STAC & RASTER REDUCTION
# ==============================================================================
def check_xarray_nan(obj):
    if isinstance(obj, xr.Dataset):
        return False
    elif pd.isna(obj):  # Handles float('nan'), np.nan, and None safely
        return True
    else:
        raise 

    
# This needn't be called for every fields, I am calling it for a sample field. 
def get_utm_crs(gdf: gpd.GeoDataFrame) -> str:
    """Determine the local UTM EPSG code from a GeoDataFrame centroid."""
    centroid = gdf.to_crs("EPSG:4326").union_all().centroid
    lon, lat = centroid.x, centroid.y
    zone = int((lon + 180) // 6) + 1
    epsg = 32600 + zone if lat >= 0 else 32700 + zone
    return f"EPSG:{epsg}"

#########################################################33    
def get_sentinel1_STAC(
        geom: shapely.geometry,
        catalog_client: pystac_client.Client,
        utm_crs: str, 
        input_bands :list, 
        start : str,
        end: str, 
        resolution: float = 10.0,
        orbit_direction: str = None,  # Optional: "ascending" or "descending"
):
    """Stream optical bands for a single event directly into memory and return
                Object stored as Xarray Dataset. 
                Xarray is 3 dimentions with x y and time, with '4th dimension' being bands. 
        """
    # 2. Prepare STAC query filters (No cloud cover filter for SAR)
    query_params = {}

    if orbit_direction in ["ascending", "descending"]:
        query_params["sat:orbit_state"] = {"eq": orbit_direction}

    # 3. STAC Search directly in memory
    search = catalog_client.search(
        collections=["sentinel-1-rtc"],
        intersects=geom.__geo_interface__,
        datetime=f"{start}/{end}",
        query=query_params if query_params else None,
        limit=100,
    )
    items = list(search.items())

    if not items:
        return np.nan

    # Sign asset URLs for Planetary Computer Azure SAS authentication
    items = [planetary_computer.sign(item) for item in items]

    # 4. Project polygon to local UTM for accurate 10m spatial rasterization
    event_gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:4326") 
    event_utm = event_gdf.to_crs(utm_crs)
    minx, miny, maxx, maxy = event_utm.total_bounds
    
     
    # 5. Load Sentinel-1 backscatter assets into memory using odc.stac
    xarray_ds = odc.stac.load(
        items,
        bands=input_bands,  # Usually ['vv', 'vh']
        crs=utm_crs,
        resolution=resolution,
        x=(minx - 20, maxx + 20),
        y=(miny - 20, maxy + 20),
        chunks={},
    ).load()  # Materialize arrays into RAM

    if xarray_ds.sizes.get("time", 0) == 0:
        return np.nan
    #print("Min ",np.array(xarray_ds['vv']).min()," Mean  ",np.array(xarray_ds['vv']).mean())
    # Convert linear amplitude values to float32
    for rawband in list(xarray_ds.data_vars): 
            xarray_ds[rawband] = xarray_ds[rawband].astype("float32")
            xarray_ds[rawband] = xarray_ds[rawband].where(xarray_ds[rawband] > -50.0)

    return xarray_ds
######################################################################################################
def get_sentinel2_STAC(
        geom: shapely.geometry,
        catalog_client: pystac_client.Client, 
        utm_crs: str, 
        input_bands :list, 
        start : str,
        end: str, 
        max_cloud_cover: int = 90,
        resolution: float = 10.0,
    ):
    """Stream optical bands for a single event directly into memory and return
            Object stored as Xarray Dataset. 
            Xarray is 3 dimentions with x y and time, with '4th dimension' being bands. 
    """

    # 2. STAC Search directly in memory
    search = catalog_client.search(
        collections=["sentinel-2-l2a"],
        intersects=geom.__geo_interface__,
        datetime=f"{start}/{end}",
        query={"eo:cloud_cover": {"lte": max_cloud_cover}},
        limit=100
        )
    items = list(search.items())
    if not items:
            return np.nan
    # Sign asset URLs for Planetary Computer Azure SAS authentication
    items = [planetary_computer.sign(item) for item in items]

    # 3. Project polygon to local UTM for accurate 10m spatial rasterization
    event_gdf = gpd.GeoDataFrame(geometry=[geom], crs="EPSG:4326")
    event_utm = event_gdf.to_crs(utm_crs)
    minx, miny, maxx, maxy = event_utm.total_bounds
    ####width, height, area = maxx-minx , maxy-miny , event_utm.geometry.area.iloc[0]/1_000_000 
    ####print("Width, Height -- ", width,height,"Area -- ",area)
    # 4. Calculate required bands and stream  directly into an xarray DataArray
    # (xarray chunks/streams directly from Azure Blob Storage COGs)
    xarray_ds = odc.stac.load(
        items,
        bands=input_bands,
        crs=utm_crs,
        resolution=resolution,
        x=(minx - 20, maxx + 20),
        y=(miny - 20, maxy + 20),
        chunks={}
    ).load() # Materialize arrays into RAM

    if xarray_ds.sizes.get("time", 0) == 0:
        return np.nan
        
    # 5. Apply ESA baseline offset correction if needed (>4000 DN)
    for rawband in list(xarray_ds.data_vars): 
        xarray_ds[rawband] = xarray_ds[rawband].astype("float32")

    return xarray_ds
    
    


1. Aggregations need to happen for raw bands too
2. 

In [4]:

RUN_ACQUIRE_PATH = Path("/home/azureuser/EBRD_Tillage/EBRD_tillage_monitoring-main/src/acquire/run_acquire.py")  
TEMP_DIR = Path("/home/azureuser/EBRD_Tillage/data/temp")
TEMP_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DATA_PATH= Path("/home/azureuser/EBRD_Tillage/data/input/EBRD_merged_classified.shp")  

# Global Variables 
stratified_limitation =-1
buffer_days = 5
resolution = 10
target_to_use = 'Nominal' # Options -- baseline (Classifica column), baseline_other(other class included), 
                                            # alex , alex_binary ,
                                            # Nominal, 
                                            # [NOT IMPELEMETD => ]
                                            ## Alex sustainability Score 
                                            #### This is currently approximate. In depth feature extraction with given depth, turnover etc need to be considered. 
use_saved = False # If selected, MPC will not be called and MLDF will be selected from saved local storage. 


kind = 'Sentinel2' # Kinda useless now, only Sentinel 2 in VM
bands_s2 = ['NDVI','NDTI','STI']
padding = 5
tempAg = ['min','mean','median']
spatAg = ['mean','median']
max_cloud_cover = 90
get_all_raw_bands_s2 = True 

use_raw_bands_also = True # For aggregations

if(get_all_raw_bands_s2): 
    input_bands_s2 = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
else: 
    input_bands_s2 = []
    for band in bands_s2: 
        input_bands_s2 += SENTINEL_INDEX_DEPENDENCIES[band]
    input_bands_s2 = list(set(input_bands_s2))

input_bands_s1 = ['vh','vv']
bands_s1 = ['vvvh']

total_bands_s2 = input_bands_s2 + bands_s2 if use_raw_bands_also else bands_s2
total_bands_s1 = input_bands_s1 + bands_s1 if use_raw_bands_also else bands_s1

In [5]:
# ==============================================================================
# 1. SETUP & PATH CONFIGURATION
# ==============================================================================
print(f"Reading shapefile from: {INPUT_DATA_PATH}")
gdf = gpd.read_file(INPUT_DATA_PATH).drop(columns = ['index'])
gdf=gdf.to_crs("EPSG:4326")
# Quick check
print(f"Successfully loaded {len(gdf)} rows.")
gdf = gdf.rename(columns={'Beginning':'start_date',"End date":'end_date'})
print(gdf.columns)
gdf["start_date"] = pd.to_datetime(gdf["start_date"])
gdf["end_date"] = pd.to_datetime(gdf["end_date"])

gdf = classify_tillage(gdf,target_to_use)


if(stratified_limitation>0 and (not use_saved)): 
    lis = []
    for name,group in gdf.groupby('Target'): 
        if(group.shape[0] < stratified_limitation): 
            lis.append(group)
        else:
            lis.append(group.sample(stratified_limitation))
        print(sum(df.shape[0] for df in lis))
    gdf = pd.concat(lis).reset_index(drop=True)
print(f"Filtered to {len(gdf)} rows.")


catalog_client = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace
    )
sample_gdf = gdf.sample()
utm_crs = get_utm_crs(sample_gdf)
## the entire gdf should not be changed here. Something about the catalog expects 4236, I haven't checked. 



Reading shapefile from: /home/azureuser/EBRD_Tillage/data/input/EBRD_merged_classified.shp
Successfully loaded 5746 rows.
Index(['fid', 'Operation', 'Group of o', 'Machinery', 'Machine', 'start_date',
       'end_date', 'Field ID', 'Crop', 'Year', 'Area, ha', 'operation_',
       'source_fil', 'FieldAreaH', 'Completion', 'Classifica', 'geometry'],
      dtype='str')
Filtered to 5746 rows.


In [57]:
# ==============================================================================
# 2.0 CHUNKING FOR MPC CALL
# ==============================================================================
def chunk_df(df, column_name, target_chunk_size):
    """
    Adds '_chunk_bin' and 'chunk_num' columns based on equal-frequency
    quantile binning of a target column.
    """
    df = df.copy()

    # Determine required number of bins
    num_chunks = int(np.ceil(len(df) / target_chunk_size))

    # Assign equal-frequency quantile intervals
    df["_chunk_bin"] = pd.qcut(df[column_name], q=num_chunks, duplicates="drop")

    # Assign integer chunk numbers (0, 1, 2, ...) ordered by the bin intervals
    df["chunk_num"] = df["_chunk_bin"].cat.codes

    return df

gdf_utm = gdf.to_crs(utm_crs)
bounds = gdf_utm.geometry.bounds
gdf_utm['width'] = bounds['maxx'] - bounds['minx']
gdf_utm['height'] = bounds['maxy'] - bounds['miny']
gdf_utm['bb_area'] = (gdf_utm['width'] * gdf_utm['height'])/1e6 # Not using gdf.geometry.area as we want axis aligned bounding box area
gdf_utm['geomarea'] = gdf_utm.geometry.area / 1e6

gdf_chunked_utm = chunk_df(gdf_utm, 'bb_area', 200) #[['chunk_num','_chunk_bin']]
gdf_chunked = gdf.copy()  ## Original gdf, unprojected 
gdf_chunked['chunk_num'] = gdf_chunked_utm['chunk_num']
gdf_chunked['_chunk_bin'] = gdf_chunked_utm['_chunk_bin']
gdf_chunked['height_utm'] = gdf_chunked_utm['height']
gdf_chunked['width_utm'] = gdf_chunked_utm['width']
gdf_chunked['bb_area_utm'] = gdf_chunked_utm['bb_area']

MAX_TARGETS_X_Y = {0 :( 207,  119),1 :( 178,  140), 2 :( 194,  166),
3 :( 231,  211),4 :( 212,  191), 5 :( 204,  169),6 :( 242,  135),
7 :( 164,  177),8 :( 183,  168), 9 :( 256,  157),10 :( 166,  195),
11 :( 183,  170),12 :( 227,  220), 13 :( 259,  193), 14 :( 318,  185),
15 :( 207,  206),16 :( 248,  187),17 :( 242,  244),18 :( 271,  183), 
19 :( 328,  224), 20 :( 229,  233), 21 :( 344,  197), 22 :( 357,  190), 
23 :( 342,  212), 24 :( 394,  292),25 :( 350,  243),26 :( 277,  327), 
27 :( 423,  224),28 :( 338,  335)}

In [7]:
# ==============================================================================
# 2.1 GET DATA FROM MPC 
# ==============================================================================
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient
import fsspec
import zarr
def get_data_from_mpc(catalog_client, gdf , target_x, target_y, utm_crs,data_kind = 'S2'): 
    """
    This function uses a Sentinel 2.
    """
    print(f"Starting in-memory processing for {len(gdf)} tillage events...")
    TARGET_CRS = utm_crs
    feature_datasets = []
    valid_feature_ids = []
    # Run in-memory extraction over events
    band_results = []
    sub_index = 0
    for idx, row in gdf.iterrows():
        #index = {'fid_key':row['fid']}
        sub_index += 1
        geom  = row['geometry']
        #start_dt = (pd.to_datetime(row["start_date"]) - pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d")
        start = pd.to_datetime(row["end_date"]).strftime("%Y-%m-%d")
        end = (pd.to_datetime(row["end_date"]) + pd.Timedelta(days=buffer_days)).strftime("%Y-%m-%d")
        print(f"Processing Event {sub_index}/{len(gdf)}: {row['fid']} start: {start} end: {end}",end="")
        if(data_kind == 'S2'):
            xarray_ds = get_sentinel2_STAC(geom, catalog_client,utm_crs,input_bands_s2, start,end, max_cloud_cover=90,resolution =10)
        elif(data_kind == 'S1'): 
            xarray_ds = get_sentinel1_STAC(geom, catalog_client,utm_crs,input_bands_s1, start,end,resolution =10,orbit_direction = None) 
        else:
            raise NotImplementedError 
        if not check_xarray_nan(xarray_ds):
                xarray_ds = xarray_ds.assign_coords(time=np.arange(xarray_ds.sizes["time"])) ##Important - changes absolute time to relative. 
                current_y, current_x = xarray_ds.sizes["y"], xarray_ds.sizes["x"]
                if(current_y > target_y or current_x > target_x):
                    raise Exception(f'Target x or Y need to be larger: Current x y- {current_x} , {current_y}; Target x,y -- {target_x} , {target_y}')
                print(' -- y,x -- ', current_y,current_x,end="")
                pad_y = max(0, target_y - current_y)
                pad_x = max(0, target_x - current_x)
                if pad_y > 0 or pad_x > 0:
                    xarray_ds = xarray_ds.pad(
                        y=(0, pad_y),
                        x=(0, pad_x),
                        constant_values=xarray_ds['B04'].attrs.get("nodata", 0))
                # Trim to exact target spatial dimensions
                xarray_ds = xarray_ds.isel(y=slice(0, target_y), x=slice(0, target_x))
                xarray_ds = xarray_ds.assign_coords(y=np.arange(target_y), x=np.arange(target_x))
                # Trigger compute for the individual polygon raster block
                xarray_ds = xarray_ds.compute()
                feature_datasets.append(xarray_ds)
                valid_feature_ids.append(row['fid'])
                print(" -- Feature Appended")
        else: 
             print(' -- Nan')

    # 4. Concatenate feature datasets along the 'feature_id' dimension
    ds_cube = xr.concat(feature_datasets, dim="feature_id")
    ds_cube["feature_id"] = valid_feature_ids
    return ds_cube  
     

In [8]:
## Shifted deleting script to findingaptsize script for now. 

In [58]:
# ==============================================================================
# 2.2 READ, and WRITE TO Datastore 
# ==============================================================================
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
# 1. Paste the exact Datastore URI copied from Azure ML Studio
uri_folder  = "azureml://subscriptions/f3ad55ae-d72a-4cce-aa07-ccefed436a79/resourcegroups/rg-rise-westeurope/workspaces/mlw-rise-westeurope/datastores/rise_data/paths/abhi/"


datastore_name = "rise_data"
datastore = ml_client.datastores.get(datastore_name)
account_name = datastore.account_name
container_name = datastore.container_name
relative_path = "abhi/TillageClassification_s2/"

##utm_crs = get_utm_crs(gdf.iloc[0:1])
##gdf =gdf.to_crs(utm_crs)

CHUNK = 0
grp = gdf_chunked[gdf_chunked.chunk_num == CHUNK]
print('READING AND WRITING CHUNK -- ',CHUNK)
target_x, target_y = MAX_TARGETS_X_Y[CHUNK]
ds_cube = get_data_from_mpc(catalog_client, grp , target_x, target_y, utm_crs)
# 3. Construct native Azure Blob storage URL
storage_url = f"az://{container_name}/{relative_path}20261006__TillageDataset__Chunk{CHUNK}"
# 4. Save directly using adlfs storage options
ds_cube.to_zarr(
    store=storage_url,
    mode="w",
    storage_options={
        "account_name": account_name,
        "credential": DefaultAzureCredential(),
    },
)
"++++++++++++++++++++++++++++++++++++++++++++++++++++++"


Found the config file in: /config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


READING AND WRITING CHUNK --  0
Starting in-memory processing for 248 tillage events...
Processing Event 1/248: 1443.0 start: 2020-09-23 end: 2020-09-28 -- y,x --  80 207 -- Feature Appended
Processing Event 2/248: 1444.0 start: 2020-09-23 end: 2020-09-28 -- y,x --  80 207 -- Feature Appended
Processing Event 3/248: 1446.0 start: 2020-09-23 end: 2020-09-28 -- y,x --  80 207 -- Feature Appended
Processing Event 4/248: 1447.0 start: 2020-09-24 end: 2020-09-29 -- y,x --  80 207 -- Feature Appended
Processing Event 5/248: 1448.0 start: 2020-09-24 end: 2020-09-29 -- y,x --  80 207 -- Feature Appended
Processing Event 6/248: 1449.0 start: 2020-09-24 end: 2020-09-29 -- y,x --  80 207 -- Feature Appended
Processing Event 7/248: 1450.0 start: 2020-09-24 end: 2020-09-29 -- y,x --  80 207 -- Feature Appended
Processing Event 8/248: 1454.0 start: 2021-04-11 end: 2021-04-16 -- Nan
Processing Event 9/248: 1455.0 start: 2021-05-13 end: 2021-05-18 -- y,x --  80 207 -- Feature Appended
Processing Event

'++++++++++++++++++++++++++++++++++++++++++++++++++++++'

In [10]:
# =============================================================================
# 2.3 READ FROM DATASTORE 
# ==============================================================================
relative_path = "abhi/TillageClassification_s2/"
storage_url = f"az://{container_name}/{relative_path}20261006__TillageDataset__Chunk1"
print(storage_url)
ds_cube_read = xr.open_zarr(
    store=storage_url,
    zarr_format=3,
    consolidated = False,
    storage_options={
        "account_name": account_name,
        "credential": DefaultAzureCredential(),
    },
)

az://data/abhi/TillageClassification_s2/20261006__TillageDataset__Chunk1


In [11]:
ds_cube_read

<xarray.Dataset> Size: 422MB
Dimensions:      (feature_id: 141, time: 3, y: 140, x: 178)
Coordinates:
  * feature_id   (feature_id) float64 1kB 5.894e+03 5.895e+03 ... 1.509e+04
  * time         (time) int64 24B 0 1 2
  * y            (y) int64 1kB 0 1 2 3 4 5 6 7 ... 133 134 135 136 137 138 139
  * x            (x) int64 1kB 0 1 2 3 4 5 6 7 ... 171 172 173 174 175 176 177
    spatial_ref  int32 4B ...
Data variables:
    B02          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>
    B03          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>
    B06          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>
    B08          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>
    B12          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>
    B04          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>
    B11          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>
    B07          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>
    B8A          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>
    B05          (feature_id, time, y, x) float32 42MB dask.array<chunksize=(36, 1, 70, 89), meta=np.ndarray>

In [34]:
import adlfs 
# Initialize the file system
fs = adlfs.AzureBlobFileSystem(
    account_name=account_name,
    credential=DefaultAzureCredential()
)

storage_url = f"az://{container_name}/{relative_path}"
print(storage_url)
# Get the total size in bytes using fs.du
total_bytes = fs.du(storage_url, total=True)

# Convert to MB and GB for readability
print(f"Total size on Azure: {total_bytes / (1024 ** 2):.2f} MB")
print(f"Total size on Azure: {total_bytes / (1024 ** 3):.3f} GB")

az://data/abhi/TillageClassification_s2/
Total size on Azure: 600.86 MB
Total size on Azure: 0.587 GB


In [ ]:
1/0

In [ ]:
old_path = "data/abhi/TillageClassification"
new_path = "data/abhi/my_dataset.zarr"

# 1. Copy all blobs to the new location (server-side copy)
#fs.copy(old_path, new_path, recursive=True)

fs.rm(new_path, recursive=True)